In [ ]:
# ================================# 04-hyperparameter-search (pooled across languages)# Single LR per method — NOT split by language# Classifier head trainable — this is the root fix# ================================!pip uninstall -y torchao!pip install -q peft --no-deps!pip install -q trl --no-deps!pip install -q accelerateimport torch, transformers, datasets, peftprint("torch:", torch.__version__)print("transformers:", transformers.__version__)print("datasets:", datasets.__version__)print("peft:", peft.__version__)print("CUDA available:", torch.cuda.is_available())print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# ================================# Imports# ================================import os, time, jsonimport pandas as pdimport numpy as npfrom transformers import AutoTokenizer, AutoModelForSequenceClassificationfrom peft import LoraConfig, IA3Config, TaskType, get_peft_modelfrom datasets import Datasetfrom torch.utils.data import DataLoaderfrom sklearn.metrics import accuracy_score, f1_scoreMODEL_NAME = "xlm-roberta-base"NUM_LABELS = 3MAX_LENGTH = 128tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)DATA_ROOT = "/kaggle/input/notebooks/venkatkolluu/02-data-preprocessingv2/data/processed"print(os.listdir(DATA_ROOT))print(os.listdir(f"{DATA_ROOT}/hi"))

In [ ]:
# ================================# Search configuration# ================================SWEEP_BUDGET = 2000LANGUAGES = ["hi", "te"]LEARNING_RATES = [5e-6, 1e-5, 5e-5, 1e-4, 5e-4, 1e-3, 5e-3]EPOCHS = 10BATCH_SIZE = 32SEEDS = [42, 123]METHODS = ["lora", "dora", "ia3"]

In [ ]:
# ================================# Data loader# ================================def build_loaders(language, budget):    train_df = pd.read_parquet(f"{DATA_ROOT}/{language}/train_{budget}.parquet")    valid_df = pd.read_parquet(f"{DATA_ROOT}/{language}/valid.parquet")    def tokenize(batch):        return tokenizer(batch["premise"], batch["hypothesis"],                          truncation=True, padding="max_length", max_length=MAX_LENGTH)    train_ds = Dataset.from_pandas(train_df).rename_column("label", "labels")    valid_ds = Dataset.from_pandas(valid_df).rename_column("label", "labels")    train_ds = train_ds.map(tokenize, batched=True)    valid_ds = valid_ds.map(tokenize, batched=True)    keep = ["input_ids", "attention_mask", "labels"]    train_ds = train_ds.remove_columns([c for c in train_ds.column_names if c not in keep])    valid_ds = valid_ds.remove_columns([c for c in valid_ds.column_names if c not in keep])    train_ds.set_format("torch"); valid_ds.set_format("torch")    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)    valid_loader = DataLoader(valid_ds, batch_size=64)    return train_loader, valid_loader

In [ ]:
# ================================# Model builder — classifier trainable (THE fix)# ================================def load_base_model():    return AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS).cuda()def build_model(method):    base = load_base_model()    if method == "lora":        config = LoraConfig(r=8, lora_alpha=16, lora_dropout=0.1, bias="none",                             task_type=TaskType.SEQ_CLS, target_modules=["query", "value"],                             modules_to_save=["classifier"])        return get_peft_model(base, config)    if method == "dora":        config = LoraConfig(r=8, lora_alpha=16, lora_dropout=0.1, bias="none", use_dora=True,                             task_type=TaskType.SEQ_CLS, target_modules=["query", "value"],                             modules_to_save=["classifier"])        return get_peft_model(base, config)    if method == "ia3":        config = IA3Config(task_type=TaskType.SEQ_CLS,                            target_modules=["key", "value", "output.dense"],                            feedforward_modules=["output.dense"],                            modules_to_save=["classifier"])        return get_peft_model(base, config)    raise ValueError(method)_test = build_model("lora")_trainable = [n for n, p in _test.named_parameters() if p.requires_grad]assert any("classifier" in n for n in _trainable), "STOP -- classifier still frozen"print("Classifier trainable -- confirmed")del _testtorch.cuda.empty_cache()

In [ ]:
# ================================# Train + evaluate one (method, language, lr, seed) config# ================================def train_and_evaluate(method, language, lr, seed, epochs=EPOCHS):    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed); np.random.seed(seed)    train_loader, valid_loader = build_loaders(language, SWEEP_BUDGET)    model = build_model(method)    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)    model.train()    for epoch in range(epochs):        for batch in train_loader:            batch = {k: v.cuda() for k, v in batch.items()}            optimizer.zero_grad()            loss = model(**batch).loss            loss.backward()            optimizer.step()    model.eval()    preds, trues = [], []    with torch.no_grad():        for batch in valid_loader:            batch = {k: v.cuda() for k, v in batch.items()}            outputs = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"])            preds.extend(torch.argmax(outputs.logits, dim=1).cpu().numpy())            trues.extend(batch["labels"].cpu().numpy())    acc = accuracy_score(trues, preds)    f1 = f1_score(trues, preds, average="macro")    del model    torch.cuda.empty_cache()    return acc, f1

In [ ]:
# ================================# Run search: 3 methods x 7 LRs x 2 langs x 2 seeds = 84 runs# ================================results = []total_configs = len(METHODS) * len(LEARNING_RATES) * len(LANGUAGES) * len(SEEDS)run_count = 0for method in METHODS:    for lr in LEARNING_RATES:        for lang in LANGUAGES:            for seed in SEEDS:                run_count += 1                print(f"[{run_count}/{total_configs}] {method} | lr={lr} | {lang} | seed={seed}")                acc, f1 = train_and_evaluate(method, lang, lr, seed)                results.append({"method": method, "lr": lr, "language": lang, "seed": seed,                                 "accuracy": acc, "macro_f1": f1})                print(f"  acc={acc:.4f}, f1={f1:.4f}")sweep_df = pd.DataFrame(results)sweep_df.to_csv("/kaggle/working/lr_sweep_raw_results.csv", index=False)

In [ ]:
# ================================# Aggregate by (method, lr), POOLED across both languages# ================================summary = sweep_df.groupby(["method", "lr"]).agg({    "accuracy": ["mean", "std"],    "macro_f1": ["mean", "std"]}).reset_index()summary.columns = ["method", "lr", "acc_mean", "acc_std", "f1_mean", "f1_std"]summary = summary.sort_values(["method", "f1_mean"], ascending=[True, False])display(summary)

In [ ]:
# ================================# Pick ONE best LR per method -- pooled across both languages# ================================best_lr_per_method = {}for method in METHODS:    sub = summary[summary["method"] == method].sort_values("f1_mean", ascending=False)    best_row = sub.iloc[0]    best_lr = float(best_row["lr"])    best_acc = best_row["acc_mean"]    best_f1 = best_row["f1_mean"]    assert best_acc > 0.35, (        f"STOP -- {method}: best LR ({best_lr}) still at/near random chance "        f"(acc={best_acc:.4f}). Check classifier is trainable, or expand LR range."    )    best_lr_per_method[method] = best_lr    print(f"{method}: best LR = {best_lr} (mean acc={best_acc:.4f}, mean f1={best_f1:.4f})")with open("/kaggle/working/chosen_lrs.json", "w") as f:    json.dump(best_lr_per_method, f, indent=2)print("Saved chosen_lrs.json:", best_lr_per_method)